In [1]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

In [2]:
data = "data/input/piebro/changeset_data/year=*/month=*/*.parquet"
db = duckdb.read_parquet(r"data/input/piebro/changeset_data/year=*/month=*/*.parquet", hive_partitioning=1) 

In [3]:
#number of distinct contributors
duckdb.sql("""
    SELECT year, COUNT(distinct user_name) as contributors
    FROM db 
    GROUP BY year
    ORDER BY year
""").show()

┌───────┬──────────────┐
│ year  │ contributors │
│ int64 │    int64     │
├───────┼──────────────┤
│  2005 │           86 │
│  2006 │          498 │
│  2007 │         5300 │
│  2008 │        29316 │
│  2009 │        73239 │
│  2010 │        79678 │
│  2011 │        95222 │
│  2012 │       123546 │
│  2013 │       128904 │
│  2014 │       156262 │
│  2015 │       163603 │
│  2016 │       262011 │
│  2017 │       318933 │
│  2018 │       308050 │
│  2019 │       281824 │
│  2020 │       302799 │
│  2021 │       292574 │
│  2022 │       258568 │
│  2023 │       273550 │
│  2024 │       263269 │
│  2025 │       273941 │
│  2026 │       109782 │
└───────┴──────────────┘
  22 rows    2 columns



In [21]:
#number of yearly changesets
duckdb.sql("""
    SELECT year, COUNT(*) as changesets
    FROM db 
    GROUP BY year
    ORDER BY year
""").show()

┌───────┬────────────┐
│ year  │ changesets │
│ int64 │   int64    │
├───────┼────────────┤
│  2005 │       1413 │
│  2006 │      16711 │
│  2007 │     138149 │
│  2008 │     505243 │
│  2009 │    2842898 │
│  2010 │    3319360 │
│  2011 │    3434023 │
│  2012 │    4228111 │
│  2013 │    5267489 │
│  2014 │    8109403 │
│  2015 │    8522272 │
│  2016 │    8654393 │
│  2017 │   10285731 │
│  2018 │   10880406 │
│  2019 │   13149744 │
│  2020 │   17685355 │
│  2021 │   18876002 │
│  2022 │   15116259 │
│  2023 │   15019265 │
│  2024 │   15113717 │
│  2025 │   15856773 │
│  2026 │    4007900 │
└───────┴────────────┘
  22 rows  2 columns



In [38]:
duckdb.sql(f"""
WITH first_edit AS (
    SELECT
        distinct user_name,
        MIN(year + month) as first_ym,  
        MIN(year)  as first_year,
        MIN(month) as first_month
    FROM db
    GROUP BY user_name
)
SELECT
    first_year  as year,
    first_month as month,
    COUNT(*) as new_contributors
    FROM first_edit
    WHERE first_year >= 2020
    GROUP BY first_year, first_month
    ORDER BY first_year, first_month
""").show()



┌───────┬───────┬──────────────────┐
│ year  │ month │ new_contributors │
│ int64 │ int64 │      int64       │
├───────┼───────┼──────────────────┤
│  2020 │     1 │            28212 │
│  2020 │     2 │            17078 │
│  2020 │     3 │            15326 │
│  2020 │     4 │            18776 │
│  2020 │     5 │            17058 │
│  2020 │     6 │            14352 │
│  2020 │     7 │            16421 │
│  2020 │     8 │            14580 │
│  2020 │     9 │            14904 │
│  2020 │    10 │            15451 │
│    ·  │     · │              ·   │
│    ·  │     · │              ·   │
│    ·  │     · │              ·   │
│  2025 │     6 │            10964 │
│  2025 │     7 │            11129 │
│  2025 │     8 │            11501 │
│  2025 │     9 │            12508 │
│  2025 │    10 │            12436 │
│  2025 │    11 │            13533 │
│  2025 │    12 │            10815 │
│  2026 │     1 │            14292 │
│  2026 │     2 │            14990 │
│  2026 │     3 │            17299 │
└

In [39]:
df = duckdb.sql(f"""
    SELECT
        user_name,
        first_year,
        first_month,
        first_changeset_id
    FROM (
        SELECT
            distinct user_name,
            MIN(year)  AS first_year,
            MIN(month) AS first_month,
            MIN(changeset_id) AS first_changeset_id
        FROM db
        GROUP BY user_name
        ORDER BY first_year, first_month
    )
""").df()

df.to_csv("new_osm_users_since_2020.csv", index=False)

In [49]:
df

,user_name,first_year,first_month,first_changeset_id
0,,2005,1,6
1,zool,2005,1,66
2,Tomcat,2005,1,156
3,kroepke,2005,1,285
4,Christian,2005,1,1269
...,...,...,...,...
2453323,Lorenzo-Cecchini,2026,3,179571001
2453324,Jedpido,2026,3,179957816
2453325,Katja Rolf,2026,3,180596948
2453326,marile perez,2026,3,179237890


In [42]:
print(f"total number of new users: {df['user_name'].nunique()}")


total number of new users: 2453328.loc[df['first_year'].isin(2025)]


In [44]:
new_user_23_df = df.loc[df['first_year'] == 2023]

In [45]:
print(f"total number of new users: {new_user_23_df['user_name'].nunique()}")

total number of new users: 158879


In [53]:
df2 = duckdb.sql("""
WITH user_first_appearance AS (
    SELECT
        user_name,
        year,
        month,
        ROW_NUMBER() OVER (PARTITION BY user_name ORDER BY year, month) as rn
    FROM (
        SELECT DISTINCT user_name, year, month
        FROM db
    )
),
first_appearances AS (
    SELECT user_name, year, month
    FROM user_first_appearance
    WHERE rn = 1
),
monthly_metrics AS (
    SELECT
        year,
        month,
        CONCAT(year, '-', LPAD(CAST(month as VARCHAR), 2, '0')) as months,
        COUNT(DISTINCT user_name) as Contributors,
        CAST(SUM(edit_count) as BIGINT) as Edits,
        CAST(COUNT(*) AS INTEGER) as Changesets
    FROM db
    GROUP BY year, month
),
monthly_new_contributors AS (
    SELECT
        year,
        month,
        COUNT(DISTINCT user_name) as "New Contributors"
    FROM first_appearances
    GROUP BY year, month
),
combined_metrics AS (
    SELECT
        m.year,
        m.month,
        m.months,
        m.Contributors,
        COALESCE(n."New Contributors", 0) as "New Contributors",
        m.Edits,
        m.Changesets
    FROM monthly_metrics m
    LEFT JOIN monthly_new_contributors n ON m.year = n.year AND m.month = n.month
)
SELECT
    months,
    Contributors,
    "New Contributors",
    Edits,
    Changesets,
    SUM("New Contributors") OVER (ORDER BY year, month) as "Accumulated Contributors",
    SUM(Edits) OVER (ORDER BY year, month) as "Accumulated Edits",
    SUM(Changesets) OVER (ORDER BY year, month) as "Accumulated Changesets"
FROM combined_metrics
ORDER BY year, month
""").df()

In [58]:
df2.to_csv("stats_total.csv")